# 02 Feature Engineering Pipeline - MPLADS Analytics
**Smart India Hackathon 2026 — Problem Statement 26102**

This notebook demonstrates the mathematical feature engineering pipeline transforming raw MPLADS records into 68 continuous, categorical, and behavioral indicators across 5 primary domain dimensions:
1. **Financial & Pacing Dynamics**: Cost deviation %, daily spend rate, tranche cadence, round-number sanctions.
2. **Timeline & Schedule Integrity**: Planned vs actual execution duration, milestone shortfall, elapsed percentage.
3. **Contractor Behavioral Signals**: Concurrency footprint, historic overrun frequency, completion track record.
4. **Geographic & Spatial Analytics**: District project load, state benchmark cost index, cross-boundary proximity.
5. **Policy & Compliance Vectors**: SC/ST population targeting, prohibited keywords matching, annual ceiling tracking.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src.pipeline.feature_engineer import engineer_features

feat_path = Path("../data/features/engineered_features.csv")
if not feat_path.exists():
    feat_path = Path("data/features/engineered_features.csv")

df_feats = pd.read_csv(feat_path, low_memory=False)
print(f"Loaded feature matrix: {df_feats.shape[0]:,} rows x {df_feats.shape[1]} engineered features.")
df_feats.head()

## 1. Feature Dimension Verification & Target Separation
Crucial architecture check: Verifying that `project_duration_days` (the target duration variable for regression) remains unscaled in natural days, while input features are scaled via empirical `reference_stats.pkl` and `scaler.pkl`.

In [ ]:
target_col = 'project_duration_days'
if target_col in df_feats.columns:
    t_series = df_feats[target_col].dropna()
    print(f"Target duration stats: Mean = {t_series.mean():.1f} days, Std = {t_series.std():.1f} days, Min = {t_series.min():.1f}, Max = {t_series.max():.1f}")
    assert t_series.var() > 1000, "Target duration variance is too low (check scaler leakage!)"
    print("Target duration target separation verified: Non-zero empirical variance preserved.")

## 2. Contractor Concurrency and Network Footprint
Evaluating contractor workload concentration across districts to detect potential delivery bottlenecks.

In [ ]:
concurrency_col = 'contractor_concurrency'
if concurrency_col in df_feats.columns:
    plt.figure(figsize=(10, 4))
    sns.countplot(x=df_feats[concurrency_col].clip(upper=10), palette="Blues_r")
    plt.title("Contractor Concurrency Distribution (Simultaneous Active Works)")
    plt.xlabel("Concurrent Public Works Sites (Capped at 10+)")
    plt.ylabel("Projects Count")
    plt.show()

## 3. Correlation Matrix for Primary Risk Predictors

In [ ]:
key_cols = [
    'cost_deviation_pct', 'days_behind_schedule', 'budget_utilization_rate',
    'progress_percentage', 'cost_per_day', 'contractor_concurrency',
    'contractor_completion_rate', 'project_duration_days'
]
valid_cols = [c for c in key_cols if c in df_feats.columns]
corr = df_feats[valid_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Cross-Correlation Matrix of Key Feature Predictors")
plt.show()